In [ ]:
import numpy as np
import torch
from torch.nn import *
class Res(Sequential):
    def forward(s, x):
        return super().forward(x)+x
class At(MultiheadAttention):
    def forward(s, x):
        return super().forward(x, x, x, attn_mask=Transformer.generate_square_subsequent_mask(x.size(-2),dtype=x.dtype,device=x.device), is_causal=True)[0]
class Swiglu(Linear):
    def __init__(s, i, o=0):
        super().__init__(i, 2*(o or i//3*8))
        s._ = Linear(o or i//3*8, i)
    def forward(s, x):
        x,_ = super().forward(x).chunk(2,-1)
        return s._(x*functional.silu(_))
RETRAIN = False
try:
    MAP_ = np.fromfile('tok.i','i').tolist()
except FileNotFoundError as _:
    print(_)
    MAP_ = [0]
    RETRAIN = True
MAP = dict(map(reversed,enumerate(MAP_)))

In [ ]:
from sqlite3 import connect
CARD = {MAP[i]:(MAP.get(alias,0),name,type&0x4802040)for i,alias,name,type in connect('cards.cdb').execute('select datas.id,alias,name,type from datas join texts on datas.id=texts.id')if i in MAP}
LF = {MAP[int(i)]:int(j)for i in open('lflist.conf').read().split('!')[1].split('\n')if'-'in i for i,j in[i.split()[:2]]if int(i)in MAP}
m = torch.load('model.pk','cpu',weights_only=False)[0].requires_grad_(False).eval()

In [ ]:
deck = {MAP[94410955]:3}
ex = sum(j for i,j in deck.items()if CARD[i][2])
main = sum(deck.values())-ex
while main<60 or ex<15:
    p = m(torch.tensor([i for i,j in deck.items()for _ in range(j)]))[-1]
    for i in deck|LF:
        if deck.get(i,0)+deck.get(CARD[i][0],0)>=LF.get(i,3):
            p[i] = p[CARD[i][0]] = -9e9
    if main==60:
        for i,(alias,name,type)in CARD.items():
            if not type:
                p[i] = p[alias] = -9e9
    elif ex==15:
        for i,(alias,name,type)in CARD.items():
            if type:
                p[i] = p[alias] = -9e9
    if main<40:
        p[0] = -9e9
    i = int(p.argmax())
    if not i:
        break
    deck[i] = deck.get(i,0)+1
    if CARD[i][2]:
        ex += 1
    else:
        main += 1
    print(CARD[i][1], main, ex)